# Main: ramas explosiva y efusiva juntas

Mismos datos, los dos tiros. No reescribe `RIconduitex5_5.py` ni `RIconduitef5_5.py`.

Cambia `CASO` (o `radius` / `Pressure` / `h2o` / `T` / `xi`) y corre todo.
Claves: `calbuco2015`, `vesuvius79`, `huaynaputina1600`, `caulle2011`, `merapi2010`, `sthelens2004`, `villarrica2015`, `pinatubo1991`, `quizapu1932`.

In [ ]:
CASO = "calbuco2015"

from casos.catalogo import CASOS, listar
from ambos_casos import _aplicar, _cargar_solvers, _ultimo, aceptada_ex, aceptada_ef
import numpy as np
import matplotlib.pyplot as plt

c = dict(CASOS[CASO])
ex, ef, cal = _cargar_solvers()
_aplicar(cal, c)
_aplicar(ex, c)
_aplicar(ef, c)

radius = c["radius1"]
Pressure = c["overP1"]
h2o = c["h2o"]
T = c["Tc"] + 273.15
xi = c["xi"]
geometry = c["geometry"]
dl = c["dl"]
pfinal = getattr(cal, "pfinal", 1.01325e5)
Rv = getattr(cal, "R", 461.11)

print(c["nombre"], "| obs:", c["estilo_obs"])
print(f"R={radius} m  dP={Pressure/1e6} MPa  H2O={h2o}  T={c['Tc']} C  xi={xi}  geom={geometry}")

## Rama explosiva (`RIconduitex5_5_f`)

Sale si fragmento o se estrangulo.

In [ ]:
[zsolex, soluex, countex, vex, rho_mex, rbubex, viscex, rho_tiex,
 fragcritex, phicritex, *restex] = ex.RIconduitex5_5_f(radius, Pressure, h2o, T, xi)

zsolex = np.asarray(zsolex)
soluex = np.asarray(soluex)
sex = _ultimo([zsolex, soluex, countex, vex, rho_mex, rbubex, viscex, rho_tiex, fragcritex, phicritex])
sex["phicrit"] = float(phicritex)
solex = (countex < 49) and aceptada_ex(sex, pfinal, Rv, T)

if geometry == "dyke":
    Qex = vex * radius * dl
else:
    Qex = vex * np.pi * radius * radius
MERex = Qex * rho_tiex
print("explosiva:", "SI" if solex else "no", "count", countex, "MER", f"{MERex:.3e}", "kg/s")

## Rama efusiva (`RIconduitef5_5_f`)

Sale si llega a $P_{atm}$ sin fragmentar.

In [ ]:
[zsolef, soluef, countef, vef, rho_mef, rbubef, viscef, rho_tief,
 fragcritef, phicritef] = ef.RIconduitef5_5_f(radius, Pressure, h2o, T, xi)

zsolef = np.asarray(zsolef)
soluef = np.asarray(soluef)
sef = _ultimo([zsolef, soluef, countef, vef, rho_mef, rbubef, viscef, rho_tief, fragcritef, phicritef])
sef["phicrit"] = float(phicritef)
solef = (countef < 49) and aceptada_ef(sef, pfinal)

if geometry == "dyke":
    Qef = vef * radius * dl
else:
    Qef = vef * np.pi * radius * radius
MERef = Qef * rho_tief
print("efusiva:", "SI" if solef else "no", "count", countef, "MER", f"{MERef:.3e}", "kg/s")

In [ ]:
print("=" * 50)
print(f"explosiva : {'SI' if solex else 'no':3}   MER={MERex:.3e} kg/s   phi={sex['phi']:.3f}   z={sex['zexit']:.1f} m")
print(f"efusiva   : {'SI' if solef else 'no':3}   MER={MERef:.3e} kg/s   phi={sef['phi']:.3f}   z={sef['zexit']:.1f} m")
if solex and solef:
    print("=> LAS DOS (no-unicidad de estilo)")
elif solex:
    print("=> solo explosiva")
elif solef:
    print("=> solo efusiva")
else:
    print("=> ninguna")

## Las dos soluciones encima (mismo estilo que `mainconduit5_5`)

In [ ]:
fig, axs = plt.subplots(1, 6, figsize=(18, 6), sharey=True)

ls_ex = "-" if solex else ":"
axs[0].semilogx(soluex[:, 0], zsolex, color="#C1440E", ls=ls_ex, lw=2, label="explosiva")
axs[1].plot(soluex[:, 1], zsolex, color="#C1440E", ls=ls_ex, lw=2)
axs[2].semilogx(soluex[:, 4], zsolex, color="#C1440E", ls=ls_ex, lw=2, label="liq explosiva")
axs[2].semilogx(soluex[:, 5], zsolex, color="#C1440E", ls="--", lw=1.5, label="gas explosiva")
axs[3].plot(soluex[:, 2], zsolex, color="#C1440E", ls=ls_ex, lw=2)
axs[4].plot(soluex[:, 3], zsolex, color="#C1440E", ls=ls_ex, lw=2)
axs[5].semilogx(viscex, zsolex, color="#C1440E", ls=ls_ex, lw=2)

ls_ef = "-" if solef else ":"
axs[0].semilogx(soluef[:, 0], zsolef, color="#0E7C7B", ls=ls_ef, lw=2, label="efusiva")
axs[1].plot(soluef[:, 1], zsolef, color="#0E7C7B", ls=ls_ef, lw=2)
axs[2].semilogx(soluef[:, 4], zsolef, color="#0E7C7B", ls=ls_ef, lw=2, label="liq efusiva")
axs[2].semilogx(soluef[:, 5], zsolef, color="#0E7C7B", ls="--", lw=1.5, label="gas efusiva")
axs[3].plot(soluef[:, 2], zsolef, color="#0E7C7B", ls=ls_ef, lw=2)
axs[4].plot(soluef[:, 3], zsolef, color="#0E7C7B", ls=ls_ef, lw=2)
axs[5].semilogx(viscef, zsolef, color="#0E7C7B", ls=ls_ef, lw=2)

axs[0].set_xlabel("Pressure (Pa)", fontweight="bold", fontsize=14)
axs[1].set_xlabel("gas volume fraction", fontweight="bold", fontsize=14)
axs[2].set_xlabel("velocity (m/s)", fontweight="bold", fontsize=14)
axs[3].set_xlabel("Nd", fontweight="bold", fontsize=14)
axs[4].set_xlabel("crystal content", fontweight="bold", fontsize=14)
axs[5].set_xlabel("viscosity (Pa.s)", fontweight="bold", fontsize=14)
axs[0].set_ylabel("Depth (m)", fontweight="bold", fontsize=14)
axs[0].legend(fontsize=8)
axs[2].legend(fontsize=7)

txt = []
if solex:
    txt.append(f"ex MER = {MERex:.3g} kg/s")
if solef:
    txt.append(f"ef MER = {MERef:.3g} kg/s")
if txt:
    fig.text(0.62, 0.90, "   |   ".join(txt), bbox=dict(facecolor="white", edgecolor="black"))

fig.suptitle(f"{c['nombre']}   R={radius:g} m   dP={Pressure/1e6:g} MPa   H2O={h2o} wt%   T={c['Tc']:.0f} C", fontsize=12)
plt.tight_layout()
plt.show()